# EDA Bloque 4 — Validación Psicológica

```
================================================================================
OBJETIVO 1 - ANALISIS EXPLORATORIO Y DESCRIPTIVO (EDA)
BLOQUE 4: VALIDACION PSICOLOGICA
================================================================================
Datasets analizados (2):
  4.1 data.csv (DASS-42)                    (Kaggle: lucasgreenwell)
      + codebook oficial: "Depression Anxiety Stress Scales Responses.txt"
  4.2 Mental Health dataset1.csv            (Kaggle: chanchal24 / input)

El DASS-42 es una escala clinicamente validada. La clave de subescalas y los
codigos demograficos provienen del codebook oficial incluido en el ZIP.
================================================================================
```

---
**Cómo leer este notebook.** Cada dataset se analiza con la misma plantilla de 5 pasos:

| Paso | Qué hace | Comandos clave |
|---|---|---|
| 1. Carga | Lee el CSV y muestra tamaño y variables | `pd.read_csv()` |
| 2. Calidad | Cuenta nulos y duplicados | `.isna()`, `.duplicated()` |
| 3. Univariante | Distribución de la variable de estrés | `.value_counts()`, `.describe()` |
| 4. Bivariante | Relación de cada variable con el estrés | `.corr()`, `.groupby()`, `pd.crosstab()` |
| 5. Hallazgos | Conclusiones del dataset | — |

Las gráficas se guardan automáticamente como PNG en la carpeta de figuras (`plt.savefig`).


**Configuración.** Importa las librerías (`pandas` para datos, `matplotlib`/`seaborn` para gráficas), define las rutas de entrada (`DATA`) y salida de figuras (`FIG`) y crea la carpeta de figuras si no existe (`os.makedirs`). `SEP` es solo una línea decorativa para separar secciones en la salida.

In [27]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ── Configuracion global ──────────────────────────────────────────────────────
BASE = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta"
ALMC = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta\Entrega\Objetivo1"
DATA = os.path.join(BASE, "Bloque 4") + os.sep
FIG  = os.path.join(ALMC, "figurasObje1") + os.sep
os.makedirs(FIG, exist_ok=True)   # crea la carpeta de figuras1 si no existe

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 140)

SEP = "=" * 80

## DATASET 4.1 — DASS-42 (lucasgreenwell, n=39.775)

In [28]:
print(SEP); print("DATASET 4.1 - DASS-42 (lucasgreenwell)"); print(SEP)

DATASET 4.1 - DASS-42 (lucasgreenwell)


**Paso 1 – Carga e inventario.** `pd.read_csv()` carga el archivo en un DataFrame (tabla en memoria); `.shape` muestra filas × columnas. Se lee con `sep='\t'` porque el archivo usa tabulaciones como separador, no comas.

In [29]:
# PASO 1: Carga (separador TAB, no coma) --------------------------------------
df = pd.read_csv(DATA + "4 Depression Anxiety Stress Scales Responses/data.csv",
                 sep="\t")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print("42 items (respuesta A, tiempo E, posicion I) + TIPI + demograficos")

Dimensiones: 39775 filas x 172 columnas
42 items (respuesta A, tiempo E, posicion I) + TIPI + demograficos


**Paso 2 – Calidad.** `.isna().sum()` cuenta valores vacíos por columna y `.duplicated().sum()` cuenta filas repetidas. El filtro de edad 13-90 elimina valores imposibles (p. ej. quien escribió su año de nacimiento en lugar de su edad).

In [30]:
# PASO 2: Calidad - limpieza de edades invalidas -------------------------------
# El codebook confirma que 'age' es respuesta libre: hay valores tipo 1998 (ano)
n_before = len(df)
df = df[(df["age"] >= 13) & (df["age"] <= 90)].copy()
print(f"Filtrado de edad a rango 13-90: {n_before} -> {len(df)} "
      f"({n_before - len(df)} registros eliminados)")

Filtrado de edad a rango 13-90: 39775 -> 39767 (8 registros eliminados)


**Paso 3 – Univariante.** `.value_counts()` cuenta cuántos registros hay en cada nivel de la variable objetivo; `.describe()` resume media, desviación y extremos. La función `subescala()` suma las respuestas de los 14 ítems que el codebook oficial asigna a cada dimensión (Depresión, Ansiedad, Estrés).

In [31]:
# PASO 3: Calculo de subescalas (clave oficial DASS-42 del codebook) ----------
dep_items = [3, 5, 10, 13, 16, 17, 21, 24, 26, 31, 34, 37, 38, 42]
anx_items = [2, 4, 7, 9, 15, 19, 20, 23, 25, 28, 30, 36, 40, 41]
str_items = [1, 6, 8, 11, 12, 14, 18, 22, 27, 29, 32, 33, 35, 39]

def subescala(items):
    """Suma las respuestas (QnA) de los items de una subescala. Rango 14-56."""
    return df[[f"Q{i}A" for i in items]].sum(axis=1)

df["dep_score"] = subescala(dep_items)
df["anx_score"] = subescala(anx_items)
df["str_score"] = subescala(str_items)

# Subconjunto proxy de poblacion estudiantil: 18-25 anos
stu = df[(df["age"] >= 18) & (df["age"] <= 25)].copy()
print(f"\nSubconjunto proxy estudiantil (18-25 anos): n={len(stu)}")
print("\nEstadisticos de las subescalas (subset 18-25):")
print(stu[["dep_score", "anx_score", "str_score"]].describe().round(1).loc[
      ["mean", "std", "50%"]])


Subconjunto proxy estudiantil (18-25 anos): n=23039

Estadisticos de las subescalas (subset 18-25):
      dep_score  anx_score  str_score
mean       35.0       30.4       35.2
std        12.0        9.9       10.3
50%        35.0       29.0       35.0


**Paso 4 – Bivariante.** `.corr()` calcula la correlación de Pearson (-1 a +1) entre cada variable y la de estrés: cerca de 0 = sin relación lineal; |r| alto = relación fuerte. `.map({'Low':0,...})` convierte la categoría de estrés a números (codificación ordinal) para poder calcular correlaciones. `.groupby()` compara la media de las variables entre grupos.

La celda también genera y guarda la figura correspondiente (`plt.savefig`).

In [32]:
# PASO 4: Desagregacion demografica (codigos del codebook oficial) ------------
stu["genero"] = stu["gender"].map({1: "Hombre", 2: "Mujer", 3: "Otro"})
stu["educacion"] = stu["education"].map({1: "Menos que secundaria",
                                          2: "Secundaria",
                                          3: "Grado universitario",
                                          4: "Posgrado"})
print("\nSubescalas medias por genero:\n",
      stu.groupby("genero")[["dep_score", "anx_score", "str_score"]].mean().round(1))
print("\nSubescalas medias por nivel educativo:\n",
      stu.groupby("educacion")[["dep_score", "anx_score", "str_score"]].mean().round(1))

# Figura 4.1a - distribuciones de las 3 subescalas
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
nombres = {"dep_score": "Depresión", "anx_score": "Ansiedad", "str_score": "Estrés"}
for ax, (col, name) in zip(axes, nombres.items()):
    sns.histplot(stu[col], bins=21, ax=ax, color="#1F5C99")
    ax.axvline(stu[col].mean(), color="#C44E52", ls="--", lw=1.5,
               label=f"media={stu[col].mean():.1f}")
    ax.set_title(f"DASS — {name}"); ax.set_xlabel("Puntuación (14-56)")
    ax.legend()
fig.suptitle(f"Subescalas DASS-42, subset 18-25 años (n={len(stu)})", y=1.03)
plt.tight_layout()
plt.savefig(FIG + "b4_1_dass_subescalas.png", dpi=130, bbox_inches="tight")
plt.close()

# Figura 4.1b - desagregacion demografica
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
stu.groupby("genero")[["dep_score", "anx_score", "str_score"]].mean() \
   .rename(columns=nombres) \
   .plot(kind="bar", ax=axes[0], color=["#1F5C99", "#DD8452", "#55A868"])
axes[0].set_title("DASS medio por género"); axes[0].tick_params(axis="x", rotation=0)
axes[0].set_ylabel("Puntuación media")
orden_edu = ["Menos que secundaria", "Secundaria", "Grado universitario", "Posgrado"]
stu.groupby("educacion")[["dep_score", "anx_score", "str_score"]].mean() \
   .reindex(orden_edu).rename(columns=nombres) \
   .plot(kind="bar", ax=axes[1], color=["#1F5C99", "#DD8452", "#55A868"])
axes[1].set_title("DASS medio por nivel educativo")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(FIG + "b4_1_dass_demograficos.png", dpi=130)
plt.close()


Subescalas medias por genero:
         dep_score  anx_score  str_score
genero                                 
Hombre       34.4       28.1       32.8
Mujer        35.1       31.0       35.7
Otro         40.2       33.0       37.8

Subescalas medias por nivel educativo:
                       dep_score  anx_score  str_score
educacion                                            
Grado universitario        34.2       30.1       34.8
Menos que secundaria       39.1       32.9       37.5
Posgrado                   34.1       29.6       34.4
Secundaria                 35.9       30.9       35.7


**Paso 5 – Hallazgos.** Síntesis de las conclusiones del dataset (se imprime tal cual para que quede registrada en la salida del notebook).

In [33]:
# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 4.1:
 - Subset 18-25 (n=23.031): medias Dep=35.0, Anx=30.4, Str=35.2 (escala 14-56).
 - Mujeres reportan mas ansiedad (31.0) que hombres (28.1); el grupo 'Otro'
   presenta el mayor malestar en las tres subescalas.
 - A menor nivel educativo completado, mayor puntuacion en las 3 subescalas.
 - Referencia normativa robusta para contrastar los niveles de estres
   autoreportados en los Bloques 1 y 2 (sin fusionar datos).
""")


HALLAZGOS 4.1:
 - Subset 18-25 (n=23.031): medias Dep=35.0, Anx=30.4, Str=35.2 (escala 14-56).
 - Mujeres reportan mas ansiedad (31.0) que hombres (28.1); el grupo 'Otro'
   presenta el mayor malestar en las tres subescalas.
 - A menor nivel educativo completado, mayor puntuacion en las 3 subescalas.
 - Referencia normativa robusta para contrastar los niveles de estres
   autoreportados en los Bloques 1 y 2 (sin fusionar datos).



## DATASET 4.2 — MENTAL HEALTH DATASET (chanchal24, n=261.328)

In [34]:
print(SEP); print("DATASET 4.2 - MENTAL HEALTH DATASET (chanchal24)"); print(SEP)

DATASET 4.2 - MENTAL HEALTH DATASET (chanchal24)


**Paso 1 – Carga e inventario.** `pd.read_csv()` carga el archivo en un DataFrame (tabla en memoria); `.shape` muestra filas × columnas.

In [35]:
# PASO 1: Carga --------------------------------------------------------------
df2 = pd.read_csv(DATA + "4 Mental Health Dataset/Mental Health dataset1.csv")
print(f"Dimensiones: {df2.shape[0]} filas x {df2.shape[1]} columnas")

Dimensiones: 261328 filas x 17 columnas


**Paso 2 – Calidad.** `.isna().sum()` cuenta valores vacíos por columna y `.duplicated().sum()` cuenta filas repetidas.

In [36]:
# PASO 2: Calidad - columna duplicada y nulos ---------------------------------
if (df2["SocialWeakness"] == df2["SocialWeakness.1"]).all():
    df2 = df2.drop(columns=["SocialWeakness.1"])
    print("Columna 'SocialWeakness.1' eliminada (duplicado exacto).")
print("Nulos:", dict(df2.isna().sum()[df2.isna().sum() > 0]))

Columna 'SocialWeakness.1' eliminada (duplicado exacto).
Nulos: {'SelfEmployed': np.int64(4356)}


**Paso 3 – Univariante.** `.value_counts()` cuenta cuántos registros hay en cada nivel de la variable objetivo; `.describe()` resume media, desviación y extremos.

In [37]:
# PASO 3: Filtro a poblacion estudiantil --------------------------------------
stu2 = df2[df2["Occupation"] == "Student"].copy()
print(f"\nSubconjunto Occupation=='Student': n={len(stu2)}")
print("\nDistribucion IncreasingStress (estudiantes):\n",
      stu2["IncreasingStress"].value_counts())


Subconjunto Occupation=='Student': n=54207

Distribucion IncreasingStress (estudiantes):
 IncreasingStress
Maybe    20830
Yes      19444
No       13933
Name: count, dtype: int64


**Paso 4 – Bivariante.** `.corr()` calcula la correlación de Pearson (-1 a +1) entre cada variable y la de estrés: cerca de 0 = sin relación lineal; |r| alto = relación fuerte. `pd.crosstab(..., normalize='index')` construye una tabla cruzada en porcentajes: qué % de cada grupo cae en cada categoría.

La celda también genera y guarda la figura correspondiente (`plt.savefig`).

In [38]:
# PASO 4: Bivariante - estres creciente vs busqueda de tratamiento ------------
ct = pd.crosstab(stu2["IncreasingStress"], stu2["Treatment"],
                 normalize="index") * 100
print("\n% que busca tratamiento segun estres creciente:\n", ct.round(1))
ct2 = pd.crosstab(stu2["FamilyHistory"], stu2["Treatment"], normalize="index") * 100
print("\n% que busca tratamiento segun antecedentes familiares:\n", ct2.round(1))

# Figura 4.2
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ct.plot(kind="bar", stacked=True, ax=axes[0], color=["#CCCCCC", "#1F5C99"])
axes[0].set_title(f"Tratamiento según estrés creciente\n(estudiantes, n={len(stu2)})")
axes[0].set_ylabel("%"); axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Tratamiento", loc="lower right")
ct2.plot(kind="bar", stacked=True, ax=axes[1], color=["#CCCCCC", "#C44E52"])
axes[1].set_title("Tratamiento según antecedentes familiares")
axes[1].set_ylabel("%"); axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Tratamiento", loc="lower right")
plt.tight_layout()
plt.savefig(FIG + "b4_2_chanchal_tratamiento.png", dpi=130)
plt.close()


% que busca tratamiento segun estres creciente:
 Treatment           No   Yes
IncreasingStress            
Maybe             51.6  48.4
No                49.3  50.7
Yes               50.4  49.6

% que busca tratamiento segun antecedentes familiares:
 Treatment        No   Yes
FamilyHistory            
No             64.9  35.1
Yes            27.6  72.4


**Paso 5 – Hallazgos.** Síntesis de las conclusiones del dataset (se imprime tal cual para que quede registrada en la salida del notebook).

In [39]:
# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 4.2:
 - La busqueda de tratamiento apenas varia con el estres creciente (48-51%):
   posible brecha de acceso o estigma en poblacion estudiantil.
 - Los antecedentes familiares SI elevan la busqueda de tratamiento.
 - Dataset masivo (54.207 estudiantes): apto para segmentaciones en dashboards.
""")

print(SEP)
print("EDA BLOQUE 4 COMPLETADO - 3 figuras generadas en", FIG)
print(SEP)


HALLAZGOS 4.2:
 - La busqueda de tratamiento apenas varia con el estres creciente (48-51%):
   posible brecha de acceso o estigma en poblacion estudiantil.
 - Los antecedentes familiares SI elevan la busqueda de tratamiento.
 - Dataset masivo (54.207 estudiantes): apto para segmentaciones en dashboards.

EDA BLOQUE 4 COMPLETADO - 3 figuras generadas en C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta\Entrega\Objetivo1\figurasObje1\
